# 멀티모달(이미지 입력) 실습

텍스트뿐 아니라 이미지까지 함께 입력으로 받아 답변하는 멀티모달 LLM 호출을 실습한다. OpenAI의 `gpt-4o-mini`는 이미지 URL(또는 로컬 이미지를 base64로 인코딩한 data URL)을 메시지에 넣어 호출할 수 있다.

In [ ]:
# !pip --version

pip 25.0.1 from D:\moon0902\hanwha_0902\ex0915\.venv\Lib\site-packages\pip (python 3.12)



In [ ]:
from langchain_openai import ChatOpenAI


## 1. 이미지 메시지 만들기 + 스트리밍 응답 헬퍼

아래 셀에서 이미지 입력을 처리하는 데 필요한 함수 3개를 정의한다.

- `image_url()`: 로컬 이미지 경로면 파일을 읽어 base64로 인코딩한 `data:` URL로 변환하고, 이미 `http(s)` URL이면 그대로 사용한다.
- `build_multimodal_messages()`: 텍스트 프롬프트와 이미지를 합쳐 `SystemMessage` + `HumanMessage`(멀티파트 content) 리스트를 만든다. 이미지는 `{"type": "image_url", "image_url": {"url": ...}}` 형태로 content에 들어간다.
- `stream_response()` / `stream_image_response()`: 스트리밍 응답을 토큰 단위로 받아서 바로 출력하고, 필요하면 이미지도 먼저 화면에 띄워준다.

In [ ]:
import base64
from pathlib import Path

from IPython.display import Image, display
from langchain_core.messages import AIMessageChunk, HumanMessage, SystemMessage

# temperature를 낮게 주어 설명이 흔들리지 않고 일관되게 나오도록 설정
llm = ChatOpenAI(
    temperature=0.1,
    model_name="gpt-4o-mini",
)

DEFAULT_SYSTEM_PROMPT = "You are a helpful assistant on parsing images."
DEFAULT_USER_PROMPT = "Explain the given images in-depth."


def image_url(image_source):
    """Convert a local image path to a data URL; keep HTTP URLs unchanged."""
    if str(image_source).startswith(("http://", "https://")):
        return str(image_source)

    # 로컬 파일이면 base64로 인코딩해서 data URL(data:image/...;base64,...) 형태로 변환
    image_path = Path(image_source)
    mime_type = "image/png" if image_path.suffix.lower() == ".png" else "image/jpeg"
    encoded_image = base64.b64encode(image_path.read_bytes()).decode("utf-8")
    return f"data:{mime_type};base64,{encoded_image}"


def build_multimodal_messages(image_source, user_prompt=None, system_prompt=None):
    """Build a [SystemMessage?, HumanMessage] list combining text + image for a vision LLM."""
    messages = [SystemMessage(content=system_prompt or DEFAULT_SYSTEM_PROMPT)]
    # HumanMessage의 content를 리스트로 주면 텍스트 + 이미지처럼 여러 파트를 한 메시지에 담을 수 있다
    content = [
        {"type": "text", "text": user_prompt or DEFAULT_USER_PROMPT},
        {"type": "image_url", "image_url": {"url": image_url(image_source)}},
    ]
    messages.append(HumanMessage(content=content))
    return messages


def stream_response(response, return_output=False):
    """Print a streamed LLM response token by token; optionally return the full text."""
    answer = ""
    for token in response:
        content = token.content if isinstance(token, AIMessageChunk) else token
        answer += content
        print(content, end="", flush=True)
    print()
    if return_output:
        return answer


def stream_image_response(image_source, user_prompt=None, system_prompt=None, show_image=True):
    """Show the image (if requested) and stream the model's answer about it."""
    if show_image:
        display(Image(url=image_url(image_source)))

    messages = build_multimodal_messages(image_source, user_prompt, system_prompt)
    return stream_response(llm.stream(messages))


## 2. 이미지 URL로 질의해보기

인터넷에 있는 이미지 URL을 그대로 `stream_image_response()`에 넘기면, 이미지를 화면에 띄우고 모델이 이미지를 설명하는 답변을 스트리밍으로 출력한다.

In [ ]:
# page99
IMAGE_URL = "https://t3.ftcdn.net/jpg/03/77/33/96/360_F_377339633_Rtv9I77sSmSNcev8bEcnVxTHrXB4nRJ5.jpg"

# 이미지 URL로부터 질의 (내부적으로 이미지 출력 + 스트리밍 답변 출력)
stream_image_response(IMAGE_URL)
